# Import Library

In [33]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve

# Load Dataset

In [34]:
nama_folder = "../Dataset"

# Load semua file CSV
tables = {}
for file in os.listdir(nama_folder):
    if file.endswith('.csv'):
        nama_tabel = os.path.splitext(file)[0] 
        path_file = os.path.join(nama_folder, file)
        
        tables[nama_tabel] = pd.read_csv(path_file)
        print(f"✔ {nama_tabel:<11} {len(tables[nama_tabel]):>7,} baris | {tables[nama_tabel].shape[1]} kolom")

# Assign tabel utama
df_customers = tables["customers"]
df_sellers   = tables["sellers"]
df_products  = tables["products"]
df_orders    = tables["orders"]
df_order_items = tables["order_items"]
df_product_category = tables["product_category_name_translation"]

✔ customers    99,441 baris | 5 kolom
✔ data_siap_prediksi   1,000 baris | 16 kolom
✔ geolocation 1,000,163 baris | 5 kolom
✔ orders       99,441 baris | 8 kolom
✔ order_items 112,650 baris | 7 kolom
✔ order_payments 103,886 baris | 5 kolom
✔ order_reviews  99,224 baris | 7 kolom
✔ products     32,951 baris | 9 kolom
✔ product_category_name_translation      71 baris | 2 kolom
✔ sellers       3,095 baris | 4 kolom


# Audit & Eksplorasi Data Awal

In [35]:
audit = pd.DataFrame({
    "baris": {n: len(t) for n, t in tables.items() if n in ["customers", "order_items", "sellers", "orders","product_category","products"]},
}).assign(
    kolom=lambda d: [tables[n].shape[1] for n in d.index],
    total_null=lambda d: [int(tables[n].isna().sum().sum()) for n in d.index],
)
display(audit)

print("\nNilai kosong per kolom (hanya yang > 0):")
for n in ["Customers", "order_items", "sellers", "orders","product_category","products"]:
    if n in tables: 
        nulls = tables[n].isna().sum()
        nulls = nulls[nulls > 0]
        if len(nulls):
            print(f"\n[{n}]")
            print(nulls.to_string())

,baris,kolom,total_null
customers,99441,5,0
order_items,112650,7,0
orders,99441,8,4908
products,32951,9,2449
sellers,3095,4,0



Nilai kosong per kolom (hanya yang > 0):

[orders]
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

[products]
product_id                      1
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


# Data Merging

In [36]:
# Gabung orders dengan customers
base_1 = df_orders.merge(df_customers, how='left', on='customer_id')

# Agregasi order items
df_order_items_agg = df_order_items.groupby('order_id').agg(
    freight_value=('freight_value', 'sum'),
    price=('price', 'sum'),
    seller_id=('seller_id', 'first'),
    product_id=('product_id', 'first'),
    shipping_limit_date=('shipping_limit_date', 'first')
).reset_index()

# Gabung dengan sellers
base_2 = df_order_items_agg.merge(df_sellers, how='left', on='seller_id')

# Gabung keseluruhan base
main_base_geo_and_loc = base_2.merge(base_1, how="left", on="order_id").assign(
    is_same_state=lambda d: d['customer_state'] == d['seller_state']
)

main_base_spec_products = main_base_geo_and_loc.merge(df_products, how="left", on="product_id").assign(
    product_volume_cm3=lambda d: d['product_length_cm'] * d['product_height_cm'] * d['product_width_cm']
).merge(df_product_category[['product_category_name', 'product_category_name_english']], how="left", on="product_category_name")

# Data Cleaning

In [37]:
kolom_waktu = [
    'order_approved_at', 'order_delivered_carrier_date', 'order_estimated_delivery_date',
    'order_delivered_customer_date', 'shipping_limit_date', 'order_purchase_timestamp'
]

# Konversi format datetime
for col in kolom_waktu:
    if col in main_base_spec_products.columns:
        main_base_spec_products[col] = pd.to_datetime(main_base_spec_products[col], errors='coerce')

# Buat target is_late dan hapus kolom duplikat bawaan merge
main_base_spec_products = main_base_spec_products.assign(
    is_late=lambda d: d['order_delivered_customer_date'] > d['order_estimated_delivery_date']
)
df_induk_bersih = main_base_spec_products.loc[:, ~main_base_spec_products.columns.duplicated()]

# Filter hanya pesanan selesai (delivered)
df_induk_bersih = df_induk_bersih[df_induk_bersih['order_status'] == 'delivered']

# Seleksi kolom final
main_df = df_induk_bersih[[
    'order_id', 'customer_state', 'seller_state', 'is_same_state', 'freight_value',
    'product_weight_g', 'product_volume_cm3', 'price',
    'product_category_name_english', 'order_purchase_timestamp',
    'order_approved_at', 'order_delivered_carrier_date',
    'order_estimated_delivery_date', 'order_delivered_customer_date',
    'shipping_limit_date', 'is_late',
    'customer_zip_code_prefix', 'seller_zip_code_prefix'
]].copy()

# Drop NA dan duplikat
waktu_wajib = ['order_approved_at', 'order_delivered_carrier_date', 'order_estimated_delivery_date', 'order_delivered_customer_date', 'shipping_limit_date']
main_df.dropna(subset=waktu_wajib + ['product_weight_g', 'product_volume_cm3'], inplace=True)
main_df['product_category_name_english'] = main_df['product_category_name_english'].fillna('unknown')
main_df.drop_duplicates(inplace=True)

# Feature Engineering

In [38]:
# Fitur Waktu & Durasi
main_df['purchase_dayofweek'] = main_df['order_purchase_timestamp'].dt.dayofweek
main_df['purchase_hour'] = main_df['order_purchase_timestamp'].dt.hour
main_df['estimated_delivery_days'] = (main_df['order_estimated_delivery_date'] - main_df['order_approved_at']).dt.total_seconds() / 86400
main_df['allowed_processing_days'] = (main_df['shipping_limit_date'] - main_df['order_approved_at']).dt.total_seconds() / 86400
main_df['estimated_courier_days'] = main_df['estimated_delivery_days'] - main_df['allowed_processing_days']

# Flag Kondisional
main_df['is_weekend_purchase'] = main_df['purchase_dayofweek'].isin([5, 6]).astype(int)
main_df['is_night_purchase'] = main_df['purchase_hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

# Rasio Fisik & Harga
main_df['freight_ratio'] = main_df['freight_value'] / (main_df['price'] + 0.001)
main_df['product_density'] = main_df['product_weight_g'] / (main_df['product_volume_cm3'] + 0.001)
main_df['state_route'] = main_df['customer_state'] + '_' + main_df['seller_state']

# Fitur Spasial (Haversine Distance)
geo = tables['geolocation'].groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean()

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

main_df['distance_km'] = haversine(
    main_df['customer_zip_code_prefix'].map(geo['geolocation_lat']),
    main_df['customer_zip_code_prefix'].map(geo['geolocation_lng']),
    main_df['seller_zip_code_prefix'].map(geo['geolocation_lat']),
    main_df['seller_zip_code_prefix'].map(geo['geolocation_lng'])
)

# Buang kolom mentah yang sudah tidak dipakai
kolom_buang = waktu_wajib + ['order_purchase_timestamp', 'customer_state', 'seller_state', 'customer_zip_code_prefix', 'seller_zip_code_prefix', 'purchase_dayofweek', 'purchase_hour']
main_df.drop(columns=kolom_buang, errors='ignore', inplace=True)

# Data Splitting

In [40]:
x = main_df.drop(columns=['is_late', 'order_id'])
y = main_df['is_late']
groups = main_df['order_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(x, y, groups))
x_train, x_test = x.iloc[tr].copy(), x.iloc[te].copy()
y_train, y_test = y.iloc[tr], y.iloc[te]
g_train = groups.iloc[tr]

train_full = x_train.copy()
train_full['is_late'] = y_train

# Target Encoding

In [41]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(x, y, groups))
x_train, x_test = x.iloc[tr].copy(), x.iloc[te].copy()
y_train, y_test = y.iloc[tr], y.iloc[te]
g_train = groups.iloc[tr]

train_full = x_train.copy()
train_full['is_late'] = y_train

# Target Encoding Function
def oof_encode(cat_train, y_tr, groups_tr, cat_test, smooth=20, n_splits=5):
    cat_tr = cat_train.reset_index(drop=True)
    yy = y_tr.astype(int).reset_index(drop=True)
    gg = groups_tr.reset_index(drop=True)
    prior = yy.mean()
    oof = np.full(len(cat_tr), prior)
    for a, b in GroupKFold(n_splits).split(cat_tr, yy, gg):
        s = yy.iloc[a].groupby(cat_tr.iloc[a]).agg(['sum', 'count'])
        enc = (s['sum'] + prior * smooth) / (s['count'] + smooth)
        oof[b] = cat_tr.iloc[b].map(enc).fillna(prior).values
    s = yy.groupby(cat_tr).agg(['sum', 'count'])
    enc = (s['sum'] + prior * smooth) / (s['count'] + smooth)
    return oof, cat_test.map(enc).fillna(prior).values

# Terapkan Encoding
x_train['historical_route_risk'], x_test['historical_route_risk'] = oof_encode(x_train['state_route'], y_train, g_train, x_test['state_route'])
x_train['historical_category_risk'], x_test['historical_category_risk'] = oof_encode(x_train['product_category_name_english'], y_train, g_train, x_test['product_category_name_english'])

# Finalisasi Fitur Model
x_train['is_train'] = 1
x_test['is_train'] = 0
combined = pd.concat([x_train, x_test])
combined_encoded = combined.drop(columns=['state_route', 'product_category_name_english'])

X_train_final = combined_encoded[combined_encoded['is_train'] == 1].drop(columns=['is_train'])
X_test_final = combined_encoded[combined_encoded['is_train'] == 0].drop(columns=['is_train'])

def engineer_features(df):
    df_new = df.copy()
    df_new['route_volume_risk'] = df_new['historical_route_risk'] * df_new['product_volume_cm3']
    df_new['route_weight_risk'] = df_new['historical_route_risk'] * df_new['product_weight_g']
    return df_new

X_train_engineered = engineer_features(X_train_final)
X_test_engineered = engineer_features(X_test_final)

# Modeling

In [42]:
a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1).split(X_train_engineered, y_train, g_train))
X_fit, X_val = X_train_engineered.iloc[a], X_train_engineered.iloc[b]
y_fit, y_val = y_train.iloc[a], y_train.iloc[b]

spw = np.sqrt((y_fit == 0).sum() / (y_fit == 1).sum())
lgb_eng = lgb.LGBMClassifier(
    n_estimators=3000, learning_rate=0.03, num_leaves=15,
    min_child_samples=100, reg_lambda=5, scale_pos_weight=spw,
    metric='average_precision', random_state=42, verbose=-1
)

lgb_eng.fit(
    X_fit, y_fit, 
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

# Threshold Tuning
pv = lgb_eng.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, pv)
beta = 1
f_beta = (1 + beta**2) * prec[:-1] * rec[:-1] / np.clip((beta**2 * prec[:-1]) + rec[:-1], 1e-9, None)
th = thr[np.argmax(f_beta)]

c:\Users\ASUS\anaconda3\envs\dl_gpu\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


# Evaluasi

In [43]:
pt = lgb_eng.predict_proba(X_test_engineered)[:, 1]
print(f"ROC-AUC {roc_auc_score(y_test, pt):.3f} | PR-AUC {average_precision_score(y_test, pt):.3f}")
print(classification_report(y_test, (pt >= th).astype(int), digits=3))

ROC-AUC 0.738 | PR-AUC 0.225
              precision    recall  f1-score   support

       False      0.940     0.906     0.923     17730
        True      0.244     0.343     0.285      1558

    accuracy                          0.861     19288
   macro avg      0.592     0.625     0.604     19288
weighted avg      0.884     0.861     0.871     19288



# Ekspor Artifacts

In [45]:
# Ekspor Model dan Threshold
model_artifacts = {
    'model': lgb_eng,
    'threshold': th
}
joblib.dump(model_artifacts, '../Model/late_delivery_model.pkl')

# Ekspor Kamus Target Encoding
prior = y_train.mean()
smooth = 20
s_route = y_train.groupby(train_full['state_route']).agg(['sum', 'count'])
route_map = ((s_route['sum'] + prior * smooth) / (s_route['count'] + smooth)).to_dict()

s_cat = y_train.groupby(train_full['product_category_name_english']).agg(['sum', 'count'])
cat_map = ((s_cat['sum'] + prior * smooth) / (s_cat['count'] + smooth)).to_dict()

joblib.dump({'route': route_map, 'category': cat_map, 'prior': prior}, '../Model/encoding_maps.pkl')

['../Model/encoding_maps.pkl']

# Data Power BI

In [46]:
df_powerbi = main_df.loc[X_test_final.index].head(1000).copy()
df_powerbi[['customer_state', 'seller_state']] = df_powerbi['state_route'].str.split('_', expand=True)
df_powerbi.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

kolom_api = [
    'is_same_state', 'freight_value', 'product_weight_g', 'product_volume_cm3', 'price',
    'estimated_delivery_days', 'allowed_processing_days', 'estimated_courier_days',
    'is_weekend_purchase', 'is_night_purchase', 'freight_ratio', 'product_density',
    'distance_km', 'customer_state', 'seller_state', 'product_category'
]
df_powerbi = df_powerbi[kolom_api]

jalur_simpan = r"..\Dataset\data_siap_prediksi.csv"
df_powerbi.to_csv(jalur_simpan, index=False)